# 🏥 Clinical Video Analysis — Treinamento YOLOv8
Treina 4 modelos especializados por contexto clínico:
- **Cirurgia**: detecção de sangramento
- **Consulta**: sinais de dor/desconforto
- **Fisioterapia**: análise de postura e movimento
- **Triagem**: linguagem corporal indicativa de violência

## 1. Instalar Dependências

In [13]:
import subprocess
subprocess.run(['pip', 'install', 'kagglehub', 'ultralytics', 'pyyaml'], check=True)
print('✅ Dependências instaladas')

✅ Dependências instaladas


## 2. Verificar GPU

In [14]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU disponível: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 0
else:
    print('⚠️  GPU não encontrada — usando CPU (treinamento mais lento)')
    DEVICE = 'cpu'


✅ GPU disponível: NVIDIA GeForce RTX 3070
   VRAM: 8.6 GB


## 3. Baixar Datasets via KaggleHub

In [15]:
import kagglehub, os
from concurrent.futures import ThreadPoolExecutor, as_completed

DATASETS = {
    'cirurgia':     'jphajp/ur5e-srube-nurse-surgical-instruments',
    'consulta':     'coder98/emotionpain',
    'fisioterapia': 'sulaimanmuhammed/wlu-rehabilitation-posture',
    'triagem':      'simuletic/cctv-aggressive-poses-and-fight-detection-dataset',
}

def download(item):
    context, dataset_id = item
    try:
        path = kagglehub.dataset_download(dataset_id)
        return context, path, None
    except Exception as e:
        return context, None, str(e)

paths = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(download, item): item[0] for item in DATASETS.items()}
    for future in as_completed(futures):
        context, path, error = future.result()
        if error:
            print(f'❌ {context}: {error}')
        else:
            paths[context] = path
            print(f'✅ {context} → {path}')

print(f'\n{len(paths)}/4 datasets baixados')


✅ fisioterapia → C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
✅ consulta → C:\Users\Xuanzang\.cache\kagglehub\datasets\coder98\emotionpain\versions\1
✅ triagem → C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1


100%|██████████| 520M/520M [00:18<00:00, 29.3MB/s] 

Extracting files...


✅ cirurgia → C:\Users\Xuanzang\.cache\kagglehub\datasets\jphajp\ur5e-srube-nurse-surgical-instruments\versions\1

4/4 datasets baixados


## 4. Inspecionar Estrutura dos Datasets

In [19]:
import os

for context, path in paths.items():
    print(f'\n📁 {context.upper()} — {path}')
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > 2:
            continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level <= 1:
            subindent = '  ' * (level + 1)
            imgs = [f for f in files if f.endswith(('.jpg', '.png', '.jpeg'))]
            txts = [f for f in files if f.endswith('.txt')]
            if imgs:
                print(f'{subindent}🖼️  {len(imgs)} imagens')
            if txts:
                print(f'{subindent}📝 {len(txts)} labels')



📁 FISIOTERAPIA — C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
1/
  Blurred/
    Arm Raise Correct/
    Arm Raise Incorrect/
    Knee Extension Correct/
    Knee Extension Incorrect/
    Sit To Stand Correct/
    Sit To Stand Incorrect/

📁 CONSULTA — C:\Users\Xuanzang\.cache\kagglehub\datasets\coder98\emotionpain\versions\1
1/
  Frame_Labels/
    Frame_Labels/
  Images/
    Images/

📁 TRIAGEM — C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1
1/
  Aggressive_Poses_Dataset/
    images/
    labels/

📁 CIRURGIA — C:\Users\Xuanzang\.cache\kagglehub\datasets\jphajp\ur5e-srube-nurse-surgical-instruments\versions\1
1/
  UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8/
    📝 2 labels
    test/
    train/
    valid/
  UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8-obb/
    📝 2 labels
    test/
    train/
    valid/


## 5. Gerar data.yaml para Cada Contexto

In [23]:
import yaml, os

# ── DETECTION: usa data.yaml do próprio dataset ──────────────────
yamls = {
    'cirurgia': os.path.join(
        paths['cirurgia'],
        'UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8',
        'data.yaml'
    ),
    'triagem': os.path.join(
        paths['triagem'],
        'Aggressive_Poses_Dataset',
        'data.yaml'
    ),
}

# ── CLASSIFICATION: aponta pra pasta raiz com subpastas por classe ─
classify_paths = {
    'fisioterapia': os.path.join(paths['fisioterapia'], 'Blurred'),
    'consulta':     os.path.join(paths['consulta'], 'Images', 'Images'),
}

# ── Validação ─────────────────────────────────────────────────────
print("📋 Caminhos configurados:")
for k, v in {**yamls, **classify_paths}.items():
    exists = "✅" if os.path.exists(v) else "❌"
    print(f"  {exists} {k} → {v}")


📋 Caminhos configurados:
  ✅ cirurgia → C:\Users\Xuanzang\.cache\kagglehub\datasets\jphajp\ur5e-srube-nurse-surgical-instruments\versions\1\UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8\data.yaml
  ✅ triagem → C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1\Aggressive_Poses_Dataset\data.yaml
  ✅ fisioterapia → C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1\Blurred
  ✅ consulta → C:\Users\Xuanzang\.cache\kagglehub\datasets\coder98\emotionpain\versions\1\Images\Images


## 6. Treinar os Modelos

In [21]:
from ultralytics import YOLO
import shutil, os

os.makedirs('models/yolov8_custom', exist_ok=True)
resultados = {}

# ── DETECTION: cirurgia + triagem ────────────────────────────────
for context in ['cirurgia', 'triagem']:
    print(f'\n{"="*55}\n  DETECTION: {context.upper()}\n{"="*55}')
    model = YOLO('yolov8n.pt')
    model.train(
        data=yamls[context],
        epochs=50, imgsz=640, batch=16,
        name=context, project='models/yolov8_custom',
        patience=10, augment=True, workers=8,
        device=DEVICE, exist_ok=True, verbose=False
    )
    src = f'models/yolov8_custom/{context}/weights/best.pt'
    dst = f'models/yolov8_custom/{context}.pt'
    if os.path.exists(src):
        shutil.copy(src, dst)
        resultados[context] = dst
        print(f'✅ Salvo em {dst}')
    else:
        print(f'❌ best.pt não encontrado para {context}')

# ── CLASSIFICATION: fisioterapia + consulta ───────────────────────
for context, data_path in classify_paths.items():
    print(f'\n{"="*55}\n  CLASSIFICATION: {context.upper()}\n{"="*55}')
    model = YOLO('yolov8n-cls.pt')
    model.train(
        data=data_path,
        epochs=50, imgsz=224, batch=32,
        name=context, project='models/yolov8_custom',
        patience=10, workers=8,
        device=DEVICE, exist_ok=True, verbose=False
    )
    src = f'models/yolov8_custom/{context}/weights/best.pt'
    dst = f'models/yolov8_custom/{context}.pt'
    if os.path.exists(src):
        shutil.copy(src, dst)
        resultados[context] = dst
        print(f'✅ Salvo em {dst}')
    else:
        print(f'❌ best.pt não encontrado para {context}')



  DETECTION: CIRURGIA
Ultralytics 8.4.19  Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Xuanzang\.cache\kagglehub\datasets\jphajp\ur5e-srube-nurse-surgical-instruments\versions\1\UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0

## 7. Avaliar Métricas dos Modelos

In [ ]:
from ultralytics import YOLO

print(f'{'Contexto':<15} {'mAP50':>8} {'mAP50-95':>10} {'Precisão':>10} {'Recall':>8}')
print('-' * 55)

for context, model_path in resultados.items():
    model = YOLO(model_path)
    metrics = model.val(data=yamls[context], verbose=False)
    print(
        f'{context:<15}'
        f'{metrics.box.map50:>8.3f}'
        f'{metrics.box.map:>10.3f}'
        f'{metrics.box.mp:>10.3f}'
        f'{metrics.box.mr:>8.3f}'
    )


## 8. Testar com Imagem Real

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

os.makedirs('data/samples', exist_ok=True)

for context, model_path in resultados.items():
    # Busca primeira imagem disponível do dataset
    test_img = next(
        (os.path.join(root, f)
         for root, _, files in os.walk(paths[context])
         for f in files if f.endswith(('.jpg', '.png', '.jpeg'))),
        None
    )

    if not test_img:
        print(f'⚠️  Nenhuma imagem encontrada para {context}')
        continue

    model = YOLO(model_path)
    results = model(test_img, conf=0.25)
    out_path = f'data/samples/resultado_{context}.jpg'
    results[0].save(out_path)

    print(f'\n🔍 {context.upper()}')
    print(f'   Imagem: {os.path.basename(test_img)}')
    print(f'   Detecções: {len(results[0].boxes)}')
    display(Image(filename=out_path, width=400))


## 9. Resumo Final

In [ ]:
import os

print('\n📦 MODELOS TREINADOS:')
print('-' * 45)
for context in ['cirurgia', 'consulta', 'fisioterapia', 'triagem']:
    path = f'models/yolov8_custom/{context}.pt'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ {context:<15} {path} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ {context:<15} não treinado')

print('\n🚀 Próximo passo: uvicorn app.main:app --reload')
